# Full Backtest Example

Complete walkthrough of running a backtest with the regime classifier framework.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from regime_classifier import (
    BacktestEngine, SignalGenerator,
    SyntheticDataLoader,
    CombinedRegimeClassifier,
    generate_report, create_tearsheet,
    load_config,
    run_walkforward_backtest,
    bootstrap_backtest,
    monte_carlo_simulation,
    ablation_test
)

## 1. Load Configuration

In [ ]:
config = load_config("../config/strategy.yaml")
print("Config loaded")
print(f"Initial capital: ${config.backtest.initial_capital:,}")
print(f"Commission: ${config.backtest.commission_per_contract}/contract")
print(f"Slippage: {config.backtest.slippage_ticks} ticks")

## 2. Generate Data

In [ ]:
nq_loader = SyntheticDataLoader("NQ", start_price=15000, volatility=0.01, drift=0.0002)
qqq_loader = SyntheticDataLoader("QQQ", start_price=350, volatility=0.01, drift=0.0002)

start = pd.Timestamp("2024-01-01 09:30", tz="America/New_York")
end = pd.Timestamp("2024-06-30 16:00", tz="America/New_York")

nq_bars = nq_loader.load_bars(start, end, "1min")
qqq_bars = qqq_loader.load_bars(start, end, "1min")
options_chain = pd.DataFrame()
footprint = pd.DataFrame()

print(f"Bars: {len(nq_bars)}")
print(f"NQ price range: {nq_bars['low'].min():.0f} - {nq_bars['high'].max():.0f}")

## 3. Run Backtest

In [ ]:
engine = BacktestEngine(config.to_dict())
signal_gen = SignalGenerator(config.to_dict())

result = engine.run(nq_bars, qqq_bars, options_chain, footprint, signal_gen)

print("=== BACKTEST RESULTS ===")
for key, value in result.metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

## 4. Visualize Results

In [ ]:
# Equity curve
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(result.equity_curve.index, result.equity_curve.values, linewidth=0.8)
axes[0].set_title(f"Equity Curve (Return: {result.metrics['total_return_pct']:.2f}%)")
axes[0].set_ylabel("Equity ($)")
axes[0].grid(True, alpha=0.3)

# Drawdown
cummax = result.equity_curve.cummax()
drawdown = (result.equity_curve - cummax) / cummax * 100
axes[1].fill_between(drawdown.index, drawdown.values, 0, alpha=0.5, color='red')
axes[1].set_title(f"Drawdown (Max: {result.metrics['max_drawdown_pct']:.2f}%)")
axes[1].set_ylabel("Drawdown (%)")
axes[1].grid(True, alpha=0.3)

# Returns distribution
returns = result.equity_curve.pct_change().dropna() * 100
axes[2].hist(returns, bins=50, alpha=0.7, edgecolor='white')
axes[2].set_title("Returns Distribution")
axes[2].set_xlabel("Return (%)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Trade Analysis

In [ ]:
trades = result.trades

if len(trades) > 0:
    print(f"Total trades: {len(trades)}")
    print(f"Win rate: {(trades['pnl'] > 0).mean():.2%}")
    print(f"Avg win: ${trades[trades['pnl'] > 0]['pnl'].mean():.2f}")
    print(f"Avg loss: ${trades[trades['pnl'] <= 0]['pnl'].mean():.2f}")
    print(f"Profit factor: {trades[trades['pnl'] > 0]['pnl'].sum() / abs(trades[trades['pnl'] <= 0]['pnl'].sum()):.2f}")
    
    # By signal type
    if 'signal_type' in trades.columns:
        print("\nBy signal type:")
        for sig, group in trades.groupby('signal_type'):
            wr = (group['pnl'] > 0).mean()
            print(f"  {sig}: {len(group)} trades, WR={wr:.2%}, Avg P&L=${group['pnl'].mean():.2f}")
    
    # By exit reason
    print("\nBy exit reason:")
    for reason, group in trades.groupby('exit_reason'):
        print(f"  {reason}: {len(group)} trades, Avg P&L=${group['pnl'].mean():.2f}")

## 6. Walk-Forward Analysis

In [ ]:
wf_results = run_walkforward_backtest(
    engine, nq_bars, qqq_bars, options_chain, footprint,
    train_months=2,
    test_months=1,
    step_months=1
)

print(f"Walk-forward periods: {len(wf_results)}")
for i, r in enumerate(wf_results):
    print(f"  Period {i+1}: Return={r.metrics['total_return_pct']:.2f}%, "
          f"Trades={r.metrics['total_trades']}, "
          f"WinRate={r.metrics['win_rate_pct']:.1f}%, "
          f"Sharpe={r.metrics['sharpe_ratio']:.2f}")

## 7. Bootstrap Validation

In [ ]:
bs_result = bootstrap_backtest(
    engine, nq_bars, qqq_bars, options_chain, footprint,
    n_paths=500,
    block_size=20,
    seed=42
)

print(f"Profit Probability: {bs_result.profit_probability:.2%}")
print(f"P-value: {bs_result.p_value:.4f}")
print(f"Mean Return: {bs_result.mean_return:.2f} pts")
print(f"Median Return: {bs_result.percentile_50:.2f} pts")
print(f"95% CI: [{bs_result.confidence_interval[0]:.2f}, {bs_result.confidence_interval[1]:.2f}]")

## 8. Monte Carlo Simulation

In [ ]:
if len(trades) > 0:
    mc_result = monte_carlo_simulation(trades, n_simulations=2000, seed=42)
    
    print(f"Profit Probability: {mc_result['profit_probability']:.2%}")
    print(f"Median Final Equity: ${mc_result['median_final_equity']:,.2f}")
    print(f"5th Percentile: ${mc_result['percentile_5']:,.2f}")
    print(f"95th Percentile: ${mc_result['percentile_95']:,.2f}")
    print(f"Median Max DD: {mc_result['median_max_drawdown']:.2%}")
    print(f"Worst Case DD (5%): {mc_result['worst_case_dd']:.2%}")

## 9. Ablation Test

In [ ]:
def disable_poc(config):
    config = config.copy()
    config["poc_reversion"] = {"enabled": False}
    return config

def disable_vwap(config):
    config = config.copy()
    config["vwap_bands"] = {"enabled": False}
    return config

def disable_break(config):
    config = config.copy()
    config["break_trades"] = {"enabled": False}
    return config

ablation_results = ablation_test(engine, nq_bars, qqq_bars, options_chain, footprint, {
    "poc_reversion": disable_poc,
    "vwap_bands": disable_vwap,
    "break_trades": disable_break
})

print("Ablation Results:")
for name, metrics in ablation_results.items():
    print(f"  {name}: Return={metrics['total_return_pct']:.2f}%, "
          f"Trades={metrics['total_trades']}, "
          f"WinRate={metrics['win_rate_pct']:.1f}%, "
          f"PF={metrics['profit_factor']:.2f}, "
          f"MaxDD={metrics['max_drawdown_pct']:.2f}%")

## 10. Generate Report

In [ ]:
generate_report(result, "../reports/full_backtest_report.pdf")
print("Report generated!")